In [25]:
import numpy as np
import pandas as pd
import json
from pathlib import Path
from scipy.stats import wilcoxon

In [26]:
RESULTS_ROOT = Path("~/interactionfields/results/mode_sweep").expanduser()
metric_files = list(RESULTS_ROOT.rglob("metrics.json"))
metric_files[:20]

[PosixPath('/home/tweninge/interactionfields/results/mode_sweep/torus/waves/seed132/metrics.json'),
 PosixPath('/home/tweninge/interactionfields/results/mode_sweep/torus/waves/seed131/metrics.json'),
 PosixPath('/home/tweninge/interactionfields/results/mode_sweep/torus/waves/seed123/metrics.json'),
 PosixPath('/home/tweninge/interactionfields/results/mode_sweep/torus/waves/seed125/metrics.json'),
 PosixPath('/home/tweninge/interactionfields/results/mode_sweep/torus/waves/seed128/metrics.json'),
 PosixPath('/home/tweninge/interactionfields/results/mode_sweep/torus/waves/seed130/metrics.json'),
 PosixPath('/home/tweninge/interactionfields/results/mode_sweep/torus/waves/seed127/metrics.json'),
 PosixPath('/home/tweninge/interactionfields/results/mode_sweep/torus/waves/seed129/metrics.json'),
 PosixPath('/home/tweninge/interactionfields/results/mode_sweep/torus/waves/seed126/metrics.json'),
 PosixPath('/home/tweninge/interactionfields/results/mode_sweep/torus/waves/seed124/metrics.json'),


In [ ]:

def parse_horizon(horizon, full_h):
    hr = str(horizon)
    if hr == "full":
        return hr, full_h
    return hr, int(hr)

def get_max_numeric_horizon(metrics_paths):
    hs = set()
    for path in metrics_paths:
        with open(path) as f:
            d = json.load(f)
        for horizons in d["node_metrics"].values():
            for h in horizons.keys():
                if str(h) != "full":
                    hs.add(int(h))
    return max(hs)

MAX_H = get_max_numeric_horizon(metric_files)
FULL_H = MAX_H + 1

def parse_metrics_file(path: Path):
    with open(path) as f:
        d = json.load(f)

    graph = d["graph"]["kind"]
    simulator = d["simulator"]["kind"]
    seed = d["seed"]
    num_nodes = d["num_nodes"]
    t_bins = d["t_bins"]
    cut = d.get("cut")

    run_id = f"{graph}|{simulator}|{seed}"

    node_rows = []
    edge_rows = []

    # -------- nodes --------
    for mode, horizons in d["node_metrics"].items():
        for horizon, m in horizons.items():
            hr, h = parse_horizon(horizon, FULL_H)

            node_rows.append({
                "run_id": run_id,
                "graph": graph,
                "simulator": simulator,
                "seed": seed,
                "mode": mode,
                "horizon_raw": hr,
                "h": h,
                "log_nmse": np.log(m["nmse"]) if (m["nmse"] is not None and m["nmse"] > 0) else np.nan,
                "nmse": m["nmse"],
                "node_corr_mean": m.get("node_corr_mean"),
                "node_corr_median": m.get("node_corr_median"),
                "num_nodes": num_nodes,
                "t_bins": t_bins,
            })

    

    # -------- edges --------
    for m in d.get("edge_metrics", []):
        edge_rows.append({
            "run_id": run_id,
            "graph": graph,
            "simulator": simulator,
            "seed": seed,
            "method": m["method"],
            "horizon": m["horizon"],
            "k": m.get("k"),
            "topk_rate": m.get("topk_rate"),
            "p_at_k": m.get("p_at_k"),
            "r_at_k": m.get("r_at_k"),
            "f1_at_k": m.get("f1_at_k"),
            "true_pos_rate": m.get("true_pos_rate"),
            "brier": m.get("brier"),
            "logloss": m.get("logloss"),
            "split": m.get("split"),
            "cut": cut,
        })

    return node_rows, edge_rows


In [ ]:


node_rows = []
edge_rows = []

for path in metric_files:
    nr, er = parse_metrics_file(path)
    node_rows.extend(nr)
    edge_rows.extend(er)

df_nodes = pd.DataFrame(node_rows)
df_edges = pd.DataFrame(edge_rows)

Max numeric h: 12 Full mapped to: 13


NameError: name 'horizon' is not defined

Q1. Can IF-derived edges outperform static baselines?

So the primary metrics are:
* precision@k (p_at_k)
* recall@k (r_at_k)
* true_pos_rate (often redundant with recall@k)
* F1@k (as a summary)

We ignore for now:
* ROC AUC / PR AUC (often NA, misleading with extreme imbalance)
* logloss / brier (only meaningful for calibrated probabilistic methods

In [29]:
df_edges["family"] = df_edges["method"].apply(
    lambda m: "if" if m.startswith("if-") else "static"
)
dfq = df_edges[
    (df_edges["split"] == "holdout") &
    (df_edges["horizon"].isin([3, 6, 9, 12]))  # or all, but be consistent
].copy()

In [30]:
agg = (
    dfq
    .groupby(["graph", "simulator", "seed", "method", "family", "horizon"])
    .agg(
        p_at_k=("p_at_k", "mean"),
        r_at_k=("r_at_k", "mean"),
        f1_at_k=("f1_at_k", "mean"),
        tpr=("true_pos_rate", "mean"),
    )
    .reset_index()
)

In [31]:
agg.groupby("family")[["p_at_k", "r_at_k", "f1_at_k"]].describe()


p_at_k                                                         r_at_k  \
         count      mean       std  min  25%  50%       75%       max   count   
family                                                                          
if       700.0  0.018707  0.057661  0.0  0.0  0.0  0.000000  0.479167   700.0   
static  4900.0  0.039881  0.083999  0.0  0.0  0.0  0.047867  0.583333  4900.0   

                  ...                f1_at_k                                \
            mean  ...       75%  max   count      mean       std  min  25%   
family            ...                                                        
if      0.071931  ...  0.000000  1.0   700.0  0.016938  0.040907  0.0  0.0   
static  0.249520  ...  0.491302  1.0  4900.0  0.051280  0.085652  0.0  0.0   

                                 
        50%       75%       max  
family                           
if      0.0  0.000000  0.347826  
static  0.0  0.082603  0.449123  

[2 rows x 24 columns]

In [32]:
wide = (
    agg
    .pivot_table(
        index=["graph", "simulator", "seed", "horizon"],
        columns="family",
        values="p_at_k"
    )
    .dropna()
)
wide["delta_p_at_k"] = wide["if"] - wide["static"]

stat, p = wilcoxon(wide["delta_p_at_k"])
print("median Δp@k:", wide["delta_p_at_k"].median(), "p:", p)

wide = (
    agg
    .pivot_table(
        index=["graph", "simulator", "seed", "horizon"],
        columns="family",
        values="r_at_k"
    )
    .dropna()
)
wide["delta_r_at_k"] = wide["if"] - wide["static"]

stat, p = wilcoxon(wide["delta_r_at_k"])
print("median Δr@k:", wide["delta_r_at_k"].median(), "p:", p)

wide = (
    agg
    .pivot_table(
        index=["graph", "simulator", "seed", "horizon"],
        columns="family",
        values="f1_at_k"
    )
    .dropna()
)
wide["delta_f1_at_k"] = wide["if"] - wide["static"]

stat, p = wilcoxon(wide["delta_f1_at_k"])
print("median Δf1@k:", wide["delta_f1_at_k"].median(), "p:", p)

median Δp@k: -0.020139695444573488 p: 9.719147991653011e-70
median Δr@k: -0.19404125286478227 p: 1.0602628725094722e-65
median Δf1@k: -0.031527585030675986 p: 4.4325423845616136e-64


Q2. Does better node prediction imply better edge recovery?

NameError: name 'df_nodes' is not defined

Q3. How does horizon affect identifiability of edges?

In [ ]:
df_edges.isna().mean().sort_values(ascending=False)


brier            0.75
logloss          0.75
graph            0.00
simulator        0.00
seed             0.00
horizon          0.00
method           0.00
topk_rate        0.00
k                0.00
f1_at_k          0.00
p_at_k           0.00
r_at_k           0.00
true_pos_rate    0.00
roc_auc          0.00
pr_auc           0.00
split            0.00
cut              0.00
dtype: float64